In [190]:
import os
from ensurepip import bootstrap
import warnings
from jupyter_builder.jupyterlab_semver import satisfies
from sklearn.impute import SimpleImputer
from optuna.trial import FixedTrial
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score, log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna
from sklearn.inspection import permutation_importance
import shap
from optuna.visualization import plot_param_importances, plot_optimization_history, plot_slice
import math




load_dotenv(find_dotenv(usecwd=True))
url = URL.create("postgresql+psycopg2",
    username=os.getenv("DB_USER"), password=os.getenv("DB_PASSWORD"),
    host="localhost", port=5433, database="airline")
engine = create_engine(url)

df = pd.read_sql("SELECT * FROM train", engine)
print(df.shape)

df['arrival_delay'] = df['arrival_delay'].fillna(df['arrival_delay'].median())
df["satisfied"] = (df["satisfaction"] == "satisfied").astype(int)
df = df.drop(columns=['satisfaction'])
df.head()
df["customer_type_code"] = (df["customer_type"] == "customer_type_code").astype(int)
df = df.drop(columns=['customer_type'])
df = pd.get_dummies(df, columns=["gender", "satisfied", "customer_type_code", "type_of_travel"], drop_first=True)
df["travel_class_code"] = (df["travel_class"] == "travel_class_code").astype(int)
df = df.drop(columns=['travel_class'])
df.head()


(103904, 25)


,idx,id,age,flight_distance,inflight_wifi,dep_arr_time_convenient,online_booking,gate_location,food_and_drink,online_boarding,...,baggage_handling,checkin_service,inflight_service,cleanliness,departure_delay,arrival_delay,gender_Male,satisfied_1,type_of_travel_Personal Travel,travel_class_code
0,0,70172,13,460,3,4,3,1,5,3,...,4,4,5,5,25,18.0,True,False,True,0
1,1,5047,25,235,3,2,3,3,1,3,...,3,1,4,1,1,6.0,True,False,False,0
2,2,110028,26,1142,2,2,2,2,5,5,...,4,4,4,5,0,0.0,False,True,False,0
3,3,24026,25,562,2,5,5,5,2,2,...,3,1,4,2,11,9.0,False,False,False,0
4,4,119299,61,214,3,3,3,3,4,5,...,4,3,3,3,0,0.0,True,True,False,0


clean data

In [191]:
x = df.drop(columns=['satisfied_1', 'idx', 'id'])
y = df['satisfied_1']

x_tmp, x_test, y_tmp, y_test = train_test_split(x, y, stratify=y, random_state=42)
x_train, x_val, y_train, y_val = train_test_split(x_tmp, y_tmp, stratify=y_tmp, random_state=42)

In [192]:
strategies = ["most_frequent", "prior", "stratified", "uniform"]
for strategy in strategies:
    dummy = DummyClassifier(strategy=strategy,random_state=42)
    dummy.fit(x_train, y_train)
    y_pred = dummy.predict(x_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, zero_division=0)
    print(f"{strategy:15s} | accuracy = {acc:.3f} | f1 = {f1:.3f}")

most_frequent   | accuracy = 0.567 | f1 = 0.000
prior           | accuracy = 0.567 | f1 = 0.000
stratified      | accuracy = 0.507 | f1 = 0.431
uniform         | accuracy = 0.505 | f1 = 0.469


start metrics baseline

In [193]:
def unic(trial, model_name):
    if model_name =="lr":
        params = {'C': trial.suggest_float("lr_c",1e-3,1e2, log=True), 'penalty': trial.suggest_categorical('lr_penalty', ['l1', 'l2']),
                  'solver': 'liblinear',
                  'random_state': 42,
                  'max_iter': 2000}
        model = Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler() ), ("clf", LogisticRegression(**params))])
    elif model_name == "xgb":
        params = {'n_estimators': 2000,
                  'max_depth': trial.suggest_int("xgb_max_depth", 3, 10),
                  'learning_rate': trial.suggest_float('xgb_learning_rate', 1e-2, 1e-1, log=True),
                  'subsample': trial.suggest_float('xgb_subsample', 0.7, 1.0),
                  'eval_metric': ['error', 'logloss', 'auc'],
                  'random_state': 42,
                  'n_jobs': -1,
                  'early_stopping_rounds': 50,
                  'reg_alpha': trial.suggest_float('xgb_reg_alpha', 1e-3, 10.0, log=True),
                  'reg_lambda': trial.suggest_float('xgb_reg_lambda', 1e-3, 10.0, log=True),
                  'tree_method': 'hist',
                  'colsample_bytree': trial.suggest_float('xgb_colsample_bytree', 0.7, 1.0),
                  'min_child_weight': trial.suggest_float('xgb_min_child_weight', 5, 20, log=True),
                  'max_bin': trial.suggest_int('xgb_max_bin', 64, 512),
                  'grow_policy': 'depthwise',
                  }
        model = XGBClassifier(**params)
    elif model_name == "lgb":
        params = {"n_estimators": 2000,
                  "learning_rate": trial.suggest_float("lgb_learning_rate", 1e-2, 1e-1, log=True),
                  "subsample": trial.suggest_float("lgb_subsample", 0.7, 1.0),
                  "subsample_freq": 1,
                  "objective": 'binary',
                  "metric": ["auc", "binary_error", "binary_logloss"],
                  "random_state": 42,
                  "n_jobs": -1,
                  "num_leaves": trial.suggest_int("lgb_num_leaves", 8,256, log=True),
                  'reg_alpha': trial.suggest_float('lgb_reg_alpha', 1e-3, 10.0, log=True),
                  'reg_lambda': trial.suggest_float('lgb_reg_lambda', 1e-4, 4.0, log=True),
                  "verbose": -1,
                  "max_bin": trial.suggest_int("lgb_max_bin", 64, 512),
                  "min_child_samples": trial.suggest_int("lgb_min_child_samples", 5,100, log=True),}
        model = LGBMClassifier(**params)
#binary и так определится самостоятельно - можно не писать (эксперемент)
    elif model_name == "cat":
        params = {"iterations": 2000,
                      "learning_rate": trial.suggest_float("cat_learning_rate", 1e-2, 1e-1, log=True),
                      "depth": trial.suggest_int("cat_depth", 4, 8),
                      "l2_leaf_reg": trial.suggest_float("cat_l2_leaf_reg", 1e-2, 10.0, log=True),
                      "random_strength": trial.suggest_float("cat_random_strength", 1e-2, 10.0, log=True),
                      "border_count": trial.suggest_int("cat_border_count", 64, 254),
                      "eval_metric": "AUC",
                  "custom_metric": ["Accuracy", "Logloss"],
                      "early_stopping_rounds": 50,
                      "random_state": 42,
                      "thread_count": -1,
                      "verbose": False,
                    "rsm": trial.suggest_float("cat_rsm", 0.6, 1.0)}
        bootstrap = trial.suggest_categorical("cat_bootstrap", ["Bayesian", "Bernoulli", "MVS"])
        params["bootstrap_type"] = bootstrap
        if bootstrap == "Bayesian":
            params["bagging_temperature"] = trial.suggest_float("cat_bag_temperature", 0.0, 1.0)
        else:
            params["subsample"] = trial.suggest_float("cat_subsample", 0.7, 1.0)
        model = CatBoostClassifier(**params)
    elif model_name == "rf":
        params = {"n_estimators": trial.suggest_int("rf_n_estimators", 200, 500),
                  "max_depth": trial.suggest_int("rf_max_depth", 5, 20),
                  "min_samples_leaf": trial.suggest_int("rf_min_samples_leaf", 5, 50),
                  "min_samples_split": trial.suggest_int("rf_min_samples_split", 2, 30),
                  "max_features": trial.suggest_float("rf_max_features", 0.6, 1.0),
                  "max_samples": trial.suggest_float("rf_max_samples", 0.6, 1.0),
                  "random_state": 42,
                  "n_jobs": -1,
                  }
        model = RandomForestClassifier(**params)
    return model

In [194]:
def fit_model(model, name, x_train, y_train, x_val, y_val):
    if name == "xgb":
        model.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
        return model.best_iteration + 1
    if name == "lgb":
        model.fit(x_train, y_train, eval_set=[(x_val, y_val)], callbacks = [lgb.early_stopping(50, first_metric_only=True, verbose = False), lgb.log_evaluation(0)],)
        return model.best_iteration_
    if name == "cat":
        model.fit(x_train, y_train, eval_set=[(x_val, y_val)])
        return model.get_best_iteration() + 1
    model.fit(x_train, y_train)
    return None

In [195]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
def make_cv(name):
    def objective(trial):
        scores, iters = [], []
        for tr_idx, val_idx in skf.split(x_tmp, y_tmp):
            xa, xb = x_tmp.iloc[tr_idx], x_tmp.iloc[val_idx]
            ya, yb = y_tmp.iloc[tr_idx], y_tmp.iloc[val_idx]
            model = unic(trial, name)
            n_trees = fit_model(model, name, xa, ya, xb, yb)
            proba = model.predict_proba(xb)[:, 1]
            scores.append(roc_auc_score(yb, proba))
            if n_trees is not None:
                iters.append(n_trees)
        if iters:
            trial.set_user_attr("best_iteration", int(np.mean(iters)))
        trial.set_user_attr("auc_std", float(np.std(scores)))
        return float(np.mean(scores))
    return objective


In [ ]:
n_trials = {"lr": 15, "xgb": 30, "lgb": 30, "cat": 20, "rf": 17}
warnings.filterwarnings("ignore", category=FutureWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)
studies = {}
for name, n in n_trials.items():
    study = optuna.create_study(direction="maximize", study_name=name,
                                sampler=optuna.samplers.TPESampler(seed=42),)
    study.optimize(make_cv(name), n_trials=n)
    studies[name] = study
    t = study.best_trial
    print(f"{name}: auc={study.best_value:.5f} +- {t.user_attrs['auc_std']:.5f}")

/home/artem/airline_file/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/home/artem/airline_file/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/home/artem/airline_file/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/home/artem/airline_file/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/home/artem/airline_file/.venv/lib/python3.12/site-packages/sklearn/linear_model

lr: auc=0.91431 +- 0.00161
xgb: auc=0.99307 +- 0.00012
lgb: auc=0.99344 +- 0.00005


In [ ]:
def build_final(name, study):
    best = study.best_trial
    model = unic(FixedTrial(best.params), name)
    n = best.user_attrs.get("best_iteration")
    if n is not None:
        n_final = int(n * 1.1)
        if name == "xgb":
            model.set_params(n_estimators=n_final, early_stopping_rounds=None)
        elif name == "lgb":
            model.set_params(n_estimators=n_final)
        elif name == "cat":
            model.set_params(iterations=n_final)
            p = model.get_params()
            p.pop("early_stopping_rounds", None)
            model = CatBoostClassifier(**p)
    return model

In [ ]:
row, final_models = [], {}
for name, study in studies.items():
    model = build_final(name, study)
    model.fit(x_tmp, y_tmp)
    proba = model.predict_proba(x_test)[:, 1]
    final_models[name] = model
    row.append({"model": name,
                "cv_auc": study.best_value,
                "test_auc": roc_auc_score(y_test, proba),
                "test_acc": accuracy_score(y_test, proba>0.5),
                "test_logloss": log_loss(y_test, proba),})
result = pd.DataFrame(row).sort_values("test_auc", ascending=False)
print(result.round(5))

In [ ]:
model = final_models["lgb"]
gain = pd.Series(model.booster_.feature_importance(importance_type="gain"),
                 index=x_tmp.columns)
gain = (gain / gain.sum() * 100).sort_values(ascending=False)

In [ ]:
perm = permutation_importance(model, x_test, y_test, scoring="roc_auc", n_jobs=-1, n_repeats=5, random_state=42)
perm_imp = pd.Series(perm.importances_mean, index=x_test.columns).sort_values(ascending=False)

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(10, 5))
gain.head(15)[::-1].plot.barh(ax=ax[0], title="Gain")
perm_imp.head(15)[::-1].plot.barh(ax=ax[1], title="Permutation")
plt.tight_layout()
plt.show()

In [ ]:
samp = x_test.sample(5000, random_state=42)
ex = shap.TreeExplainer(model)
sv = ex.shap_values(samp)
if isinstance(sv, list):
    sv = sv[1]
shap.summary_plot(sv)
shap.summary_plot(sv, samp, plot_type="bar")

In [ ]:
for name in ["xgb", "lgb", "cat"]:
    df = studies[name].trials_dataframe()
    df = df[df["state"] == "COMPLETE"]
    print(name, "best trials:", studies[name].best_trial.user_attrs["best_iteration"], "из 2k")
    print(df["user_attrs_best_iteration"].describe().round(0), "\n")

In [ ]:
df = studies["lgb"].trials_dataframe()
print(df[["params_lgb_learning_rate", "user_attrs_best_iteration", "value"]].sort_values("value", ascending=False).head(10))

In [ ]:
imp = optuna.importance.get_param_importances(studies["lgb"])
for k, v in imp.items():
    print(f"{k:28s}: {v:.4f}")

In [ ]:
plot_param_importances(studies["lgb"]).show()

In [ ]:
plot_optimization_history(studies["lgb"]).show()

In [ ]:
import math
def edge(study):
    t = study.best_trial
    for p, v in t.params.items():
        d = t.distributions[p]
        if not hasattr(d, "low"):
            continue
        lo, hi = (math.log(d.low), math.log(d.high)) if d.log else (d.low, d.high)
        x = math.log(v) if d.log else v
        pos = (x - lo) / (hi - lo)
        flag = " <-- y границы" if pos < 0.1 or pos > 0.9 else ""
        print(f"{p:28s} {v:10.4g} [{d.low:g} .. {d.high:g}]  {pos:.2f}{flag}")
edge(studies["lgb"])

In [ ]:
fig = plot_slice(studies["lgb"])
fig.show(renderer="browser")